### 🔑 Before you start: your personal lab ID

This lab environment is shared: several students use the same Unity Catalog metastore, and a catalog name can only exist once in a metastore. If everyone created a catalog called `edu_dev`, only the first student could use it — everyone else would get errors such as *User does not have BROWSE on Catalog*.

To keep your work separate, the first code cell in this notebook works out your personal **lab ID** (8 characters, for example `1a2b3c4d`) and names your catalog **`edu_dev_<lab ID>`**. Your lab ID is the same in every notebook in this course.

- You create the catalog yourself in Task 1.1. A provided cell then makes it the **current catalog**, so short names such as `silver.students` automatically point to your catalog. Wherever the instructions mention `edu_dev`, use your own catalog.
- Where a full name or path is needed, use the Python variable `catalog` (for example `f"/Volumes/{catalog}/..."`), or type the catalog name printed by the setup cell.
- If you take a break and your session restarts, or you see *SCHEMA_NOT_FOUND* / *TABLE_OR_VIEW_NOT_FOUND* errors, **run the setup cell again** and the **Make your catalog the current catalog** cell after it.

## Exercise 1: Set Up the Catalog Structure

You are a data engineer at **Lakeside University**, a mid-sized institution digitizing its academic operations. Your team is building a data platform in Azure Databricks and Unity Catalog to manage student records, course catalogues, and enrollment data.

In this exercise, you create the top-level namespace structure: one `catalog` and three `schemas` following the **medallion architecture** naming pattern.

Here's the full Unity Catalog structure you will create:

```txt

Unity Catalog
└── edu_dev_<lab ID>  [catalog]   ← your personal edu_dev catalog
    │   tags: environment=development | university=lakeside | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [managed volume]
    │       └── students.csv
    │
    ├── silver  [schema]
    │   ├── students  [table]
    │   │   ├── student_id       BIGINT  NOT NULL  ← PK
    │   │   ├── first_name       STRING
    │   │   ├── last_name        STRING
    │   │   ├── email            STRING
    │   │   ├── enrollment_year  INT
    │   │   ├── program          STRING
    │   │   └── phone_number     STRING  (added in Exercise 6)
    │   │
    │   ├── courses  [table]
    │   │   ├── course_id    BIGINT  NOT NULL  ← PK
    │   │   ├── course_name  STRING
    │   │   ├── department   STRING
    │   │   └── credits      INT
    │   │
    │   ├── enrollments  [table]
    │   │   ├── enrollment_id  BIGINT  NOT NULL  ← PK
    │   │   ├── student_id     BIGINT            → FK → students.student_id
    │   │   ├── course_id      BIGINT            → FK → courses.course_id
    │   │   ├── semester       STRING
    │   │   └── grade          DECIMAL(4,2)
    │   │
    │   ├── vw_student_enrollments  [view]
    │   │   └── joins students + courses + enrollments
    │   │
    │   └── get_grade_classification(grade)  [function]
    │       └── returns STRING  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [materialized view]
            └── aggregates enrollments + courses
                (department, total_enrollments, avg_grade, distinct_students)
```

### Task 1.1 — Create the development catalog

Create your **edu_dev** catalog with a descriptive comment. This catalog serves as the **development environment** for Lakeside University's data platform.

Because this lab environment is shared, name the catalog `edu_dev_<lab ID>`. The first cell below works out your lab ID and stores the full catalog name in the Python variable `catalog`, and the workspace storage path in `storage_root`.

**Naming convention reminders:**
- Use lowercase with underscores
- No periods, spaces, forward slashes, or control characters

> 🤖 **Genie Code tip:** Ask Genie Code:
> *"How do I create a Unity Catalog catalog in Databricks SQL with a comment?"*

**Hint:** Use the SQL command `CREATE CATALOG`. This workspace has no default storage, so add `MANAGED LOCATION '<storage root>'`. Run the statement from Python with `spark.sql(f"...")` so you can use the `catalog` and `storage_root` variables.

In [ ]:
# Lab environment setup: several students share one Unity Catalog metastore, and a
# catalog name can only exist once in a metastore. Every catalog in this course
# therefore gets your personal lab ID as a suffix. The lab ID is calculated from your
# user name and your workspace, so it is the same in every notebook.
# If your session restarts, simply run this cell again.
import hashlib

if "lab_id" not in globals():
    # Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
    # Use the current catalog to reliably find the workspace default catalog,
    # regardless of its naming convention.
    default_catalog = spark.catalog.currentCatalog()

    storage_root = (
        spark.sql(f"DESCRIBE CATALOG EXTENDED `{default_catalog}`")
        .filter("info_name = 'Storage Root'")
        .select("info_value")
        .first()[0]
    )
    current_user = spark.sql("SELECT current_user()").first()[0]
    lab_id = hashlib.sha1(f"{current_user}|{storage_root}".encode("utf-8")).hexdigest()[:8]

print (f"Storage root: {storage_root}")
print (f"Your lab ID : {lab_id}")

catalog = f"edu_dev_{lab_id}"
print (f"Your catalog: {catalog}  (you create it in the next task)")

In [ ]:
# TODO: Create your edu_dev catalog with a descriptive comment
# Use the variables from the cell above:
#   catalog      -> your catalog name (edu_dev_<lab ID>)
#   storage_root -> the MANAGED LOCATION for the catalog


### ✅ Provided: Make your catalog the current catalog

Run the cell below. It makes your `edu_dev_<lab ID>` catalog the **current catalog** for this session, so from now on short names such as `silver.students` point to your catalog and you don't need to type your lab ID in every statement.

In [ ]:
# ✅ Run this cell — makes your catalog the current catalog for this session
spark.sql(f"USE CATALOG {catalog}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")

### Task 1.2 — Create medallion schemas

Within your `edu_dev_<lab ID>` catalog, create three schemas that implement the **medallion architecture**:

| Schema | Purpose |
|--------|---------|
| `bronze` | Raw ingested data — unmodified source files |
| `silver` | Cleaned, validated, and enriched data |
| `gold` | Aggregated, analytics-ready data |

> 🤖 **Genie Code tip:** Ask:
> *"Show me how to create multiple schemas in a Databricks Unity Catalog catalog using SQL."*

**Hint:** Use `CREATE SCHEMA IF NOT EXISTS schema_name` for each schema. Because your catalog is now the current catalog, each schema is created inside it (the full name would be `edu_dev_<lab ID>.schema_name`).

In [ ]:
%sql
-- TODO: Create the bronze, silver, and gold schemas inside your catalog

## Exercise 2: Create Tables with Constraints

With the catalog structure in place, create the core tables for Lakeside University's academic data in the `silver` schema. You'll define **primary key** and **foreign key** constraints to document the data model. Note that in Unity Catalog, these constraints are informational — they help optimize queries but do not enforce referential integrity at write time.

### Task 2.1 — Create the `students` table

Create a managed Delta table `silver.students` to store student records with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `student_id` | BIGINT | Unique student identifier (primary key) |
| `first_name` | STRING | Student's first name |
| `last_name` | STRING | Student's last name |
| `email` | STRING | University email address |
| `enrollment_year` | INT | Year the student enrolled |
| `program` | STRING | Degree program |

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a managed Delta table with a primary key constraint in Unity Catalog?"*

**Hint:** Add `CONSTRAINT students_pk PRIMARY KEY (student_id)` at the end of the column list.

In [ ]:
%sql
-- TODO: Create silver.students with a primary key constraint

### Task 2.2 — Create the `courses` table

Create `silver.courses` with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `course_id` | BIGINT | Unique course identifier (primary key) |
| `course_name` | STRING | Full name of the course |
| `department` | STRING | Academic department offering the course |
| `credits` | INT | Number of academic credits |

> 🤖 **Genie Code tip:** Ask:
> *"Write a CREATE TABLE statement for a courses table in Databricks Unity Catalog with a primary key constraint."*

In [ ]:
%sql
-- TODO: Create silver.courses with a primary key constraint

### Task 2.3 — Create the `enrollments` table with foreign keys

Create `silver.enrollments` to record which students are enrolled in which courses. This table must reference both `students` and `courses` via foreign key constraints.

| Column | Type | Description |
|--------|------|-------------|
| `enrollment_id` | BIGINT | Unique enrollment record (primary key) |
| `student_id` | BIGINT | References `students.student_id` |
| `course_id` | BIGINT | References `courses.course_id` |
| `semester` | STRING | Academic semester (e.g., `Spring 2023`) |
| `grade` | DECIMAL(4,2) | Numerical grade on a 0.0–10.0 scale |

> 🤖 **Genie Code tip:** Ask:
> *"How do I add foreign key constraints referencing other tables when creating a table in Databricks Unity Catalog?"*

**Hint:** Use `CONSTRAINT <name> FOREIGN KEY (<col>) REFERENCES <catalog>.<schema>.<table>(<col>)` for each foreign key.

In [ ]:
%sql
-- TODO: Create silver.enrollments with primary key and two foreign key constraints

### Sample data — run this cell

The following cell inserts sample data into your three tables. **Run it without modification** before continuing to Exercise 3.

In [ ]:
%sql
-- Populate students
INSERT INTO silver.students VALUES
(1001, 'Emma',   'Watson',    'emma.watson@lakeside.edu',    2022, 'Computer Science'),
(1002, 'Liam',   'Ahmed',     'liam.ahmed@lakeside.edu',     2023, 'Data Science'),
(1003, 'Sofia',  'Chen',      'sofia.chen@lakeside.edu',     2021, 'Mathematics'),
(1004, 'James',  'Okonkwo',   'james.okonkwo@lakeside.edu',  2022, 'Computer Science'),
(1005, 'Aisha',  'Patel',     'aisha.patel@lakeside.edu',    2023, 'Data Science'),
(1006, 'Noah',   'Yamamoto',  'noah.yamamoto@lakeside.edu',  2020, 'Mathematics'),
(1007, 'Olivia', 'Kowalski',  'olivia.kowalski@lakeside.edu',2021, 'Computer Science'),
(1008, 'Carlos', 'Reyes',     'carlos.reyes@lakeside.edu',   2022, 'Data Science'),
(1009, 'Mei',    'Lindqvist', 'mei.lindqvist@lakeside.edu',  2023, 'Mathematics'),
(1010, 'Jake',   'OBrien',    'jake.obrien@lakeside.edu',    2020, 'Computer Science');

-- Populate courses
INSERT INTO silver.courses VALUES
(101, 'Introduction to Programming',   'Computer Science', 3),
(102, 'Data Structures',               'Computer Science', 4),
(103, 'Statistics for Data Science',   'Data Science',     3),
(104, 'Machine Learning Fundamentals', 'Data Science',     4),
(105, 'Calculus I',                    'Mathematics',      4),
(106, 'Linear Algebra',                'Mathematics',      3);

-- Populate enrollments
INSERT INTO silver.enrollments VALUES
(1,  1001, 101, 'Spring 2023', 8.50),
(2,  1001, 102, 'Fall 2023',   7.80),
(3,  1002, 103, 'Spring 2023', 9.20),
(4,  1002, 104, 'Fall 2023',   6.50),
(5,  1003, 105, 'Spring 2022', 7.00),
(6,  1003, 106, 'Fall 2022',   8.00),
(7,  1004, 101, 'Fall 2022',   6.80),
(8,  1005, 103, 'Spring 2023', 9.00),
(9,  1006, 105, 'Fall 2020',   7.50),
(10, 1007, 102, 'Spring 2022', 8.20),
(11, 1008, 104, 'Fall 2023',   7.30),
(12, 1009, 106, 'Spring 2023', 8.80),
(13, 1010, 101, 'Fall 2021',   5.50),
(14, 1004, 102, 'Spring 2023', 7.10),
(15, 1005, 104, 'Fall 2023',   8.90);

## Exercise 3: Create Views

Views provide a virtual layer over your tables, simplifying access to joined data and pre-computing expensive aggregations. In this exercise, you create a **standard view** for combined enrollment records and a **materialized view** in the `gold` schema to serve the Lakeside University analytics team with pre-aggregated department statistics.

### Task 3.1 — Create a standard view

Create a view `silver.vw_student_enrollments` that presents a combined, human-readable view of student enrollment records. It should include:

- Student full name (concatenate `first_name` and `last_name` with a space between them)
- Student `email`
- `course_name`
- `department`
- `semester`
- `grade`

> 🤖 **Genie Code tip:** Ask:
> *"Write a SQL view in Azure Databricks that joins three tables: students, courses, and enrollments"*

**Hint:** Use `CONCAT(first_name, ' ', last_name)` or the `||` operator to combine names. You need to join all three silver tables.

In [ ]:
%sql
-- TODO: Create the silver.vw_student_enrollments view
-- Join students, courses, and enrollments; include full_name, email, course_name, department, semester, grade

### Task 3.2 — Create a materialized view for department statistics

Create a materialized view `gold.vw_department_enrollment_stats` that pre-computes the following per academic department:

| Output column | Description |
|--------------|-------------|
| `department` | Academic department name |
| `total_enrollments` | Total number of enrollment records |
| `avg_grade` | Average grade, rounded to 2 decimal places |
| `distinct_students` | Number of unique students enrolled |

> 🤖 **Genie Code tip:** Ask:
> *"What is the syntax for creating a materialized view in Azure Databricks that aggregates data from two joined tables?"*

**Hint:** Use `CREATE MATERIALIZED VIEW`, join `enrollments` with `courses`, and use `COUNT()`, `ROUND(AVG(...), 2)`, and `COUNT(DISTINCT ...)`.

> [!NOTE]
> Materialized views require a SQL warehouse to run. If this cell fails, check the compute selector at the top of the notebook and switch from **Serverless** compute to a **Serverless SQL warehouse**.
>
> A SQL warehouse starts a new session, so it doesn't know your current catalog. If you switch, add `USE CATALOG edu_dev_<lab ID>;` (with your real lab ID) as the first line of this cell and of the test cell below. When you switch back to **Serverless**, run the setup cell and the `USE CATALOG` cell at the top of the notebook again.

In [ ]:
%sql
-- TODO: Create the gold.vw_department_enrollment_stats materialized view

**✅ Test your materialized view** — Run the following query to verify it returns per-department aggregates:

You should see one row per academic department with columns for `total_enrollments`, `avg_grade`, and `distinct_students`.

In [ ]:
%sql
SELECT * FROM gold.vw_department_enrollment_stats;

## Exercise 4: Create a Volume and Read Files

Volumes extend Unity Catalog governance to files, not just structured tables. In this exercise, you create a **managed volume** in the `bronze` schema to serve as a landing zone for raw CSV files — simulating files arriving from Lakeside University's external student information system.

### Task 4.1 — Create a managed volume

Create a managed volume named `raw_files` in the `bronze` schema. This will serve as the landing area for raw data files.

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a managed volume in Unity Catalog using a SQL statement?"*

**Hint:** Use `CREATE VOLUME IF NOT EXISTS <catalog>.<schema>.<volume>`.

In [ ]:
%sql
-- TODO: Create the raw_files managed volume in bronze

### Task 4.2 — Write a CSV file to the volume

The following Python cell writes a sample student CSV file directly into your new volume. **Run it without modification.**

In [ ]:
# Write a sample student CSV file to the volume — run this cell without modification
csv_content = """student_id,first_name,last_name,email,enrollment_year,program
1001,Emma,Watson,emma.watson@lakeside.edu,2022,Computer Science
1002,Liam,Ahmed,liam.ahmed@lakeside.edu,2023,Data Science
1003,Sofia,Chen,sofia.chen@lakeside.edu,2021,Mathematics
1004,James,Okonkwo,james.okonkwo@lakeside.edu,2022,Computer Science
1005,Aisha,Patel,aisha.patel@lakeside.edu,2023,Data Science
1006,Noah,Yamamoto,noah.yamamoto@lakeside.edu,2020,Mathematics
1007,Olivia,Kowalski,olivia.kowalski@lakeside.edu,2021,Computer Science
1008,Carlos,Reyes,carlos.reyes@lakeside.edu,2022,Data Science
1009,Mei,Lindqvist,mei.lindqvist@lakeside.edu,2023,Mathematics
1010,Jake,OBrien,jake.obrien@lakeside.edu,2020,Computer Science"""

dbutils.fs.put(f"/Volumes/{catalog}/bronze/raw_files/students.csv", csv_content, overwrite=True)
print(f"students.csv written to /Volumes/{catalog}/bronze/raw_files/")

### Task 4.3 — Read the CSV from the volume into a DataFrame

Read the CSV file you just placed in the volume into a Spark DataFrame and display its contents. The file is at `/Volumes/edu_dev_<lab ID>/bronze/raw_files/students.csv` — in Python you can build this path with `f"/Volumes/{catalog}/bronze/raw_files/students.csv"`.

> 🤖 **Genie Code tip:** Ask:
> *"How do I read a CSV file with a header row from a Unity Catalog volume path in PySpark?"*

**Hint:** Use `spark.read.csv(path, header=True, inferSchema=True)` then call `.display()` or `.show()` on the result.

In [ ]:
# TODO: Read students.csv from your volume into a Spark DataFrame and display it
# Path: f"/Volumes/{catalog}/bronze/raw_files/students.csv"

## Exercise 5: Create a Reusable SQL Function

Unity Catalog allows you to create **governed, reusable functions** that encapsulate business logic. Instead of repeating grade conversion logic in every query, you'll write it once as a SQL function and call it from any notebook or query in the workspace.

### Task 5.1 — Create a grade classification function

Create a SQL scalar function `silver.get_grade_classification` that accepts a `grade DECIMAL(4,2)` and returns a `STRING` letter classification based on the scale used at Lakeside University:

| Grade range | Classification |
|-------------|----------------|
| ≥ 8.5 | `'A'` |
| ≥ 7.0 | `'B'` |
| ≥ 5.5 | `'C'` |
| ≥ 4.0 | `'D'` |
| < 4.0 | `'F'` |

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a SQL scalar function in Unity Catalog that uses a CASE expression to return a letter grade based on a numeric score?"*

**Hint:** Use `CREATE FUNCTION ... RETURNS STRING RETURN CASE WHEN ... END`.

In [ ]:
%sql
-- TODO: Create the silver.get_grade_classification function

### Task 5.2 — Test the function

Query `silver.enrollments` and apply your new function to produce a `grade_classification` column for each row. Include: `enrollment_id`, `student_id`, `course_id`, `semester`, `grade`, and `grade_classification`.

> 🤖 **Genie Code tip:** Ask:
> *"How do I call a Unity Catalog user-defined function in a SELECT statement in Azure Databricks?"*

In [ ]:
%sql
-- TODO: Query silver.enrollments and call get_grade_classification on the grade column

## Exercise 6: DDL Operations

Unity Catalog objects evolve over time as requirements change. In this exercise, you use `ALTER` statements to:
1. Extend the `students` table with a new column.
2. Apply governance **tags** to your `edu_dev_<lab ID>` catalog.

### Task 6.1 — Add a column to the students table

The student registration system has been updated to capture phone numbers. Add a nullable `phone_number` column of type `STRING` to `silver.students`.

> 🤖 **Genie Code tip:** Ask:
> *"How do I add a new nullable column to an existing Delta table in Azure Databricks using SQL?"*

**Hint:** Use `ALTER TABLE`.

In [ ]:
%sql
-- TODO: Add a phone_number (STRING) column to silver.students

### Task 6.2 — Apply tags to the catalog

Apply the following tags to your `edu_dev_<lab ID>` catalog to support governance and discoverability:

| Tag key | Tag value |
|---------|----------|
| `environment` | `'development'` |
| `university` | `'lakeside'` |
| `data_classification` | `'internal'` |

> 🤖 **Genie Code tip:** Ask:
> *"How do I set metadata tags on a Unity Catalog catalog using the ALTER CATALOG statement in Databricks?"*

In [ ]:
%sql
-- TODO: Apply the three tags to your catalog using ALTER CATALOG ... SET TAGS
-- Type your full catalog name as printed by the setup cell, for example: ALTER CATALOG edu_dev_1a2b3c4d SET TAGS (...)

### Task 6.3 — Verify your work

Run the following cells to confirm your catalog structure, table schemas, and tags are all in place. Compare the output with what you created throughout this lab.

In [ ]:
%sql
-- Show all tables and views in the silver schema
SHOW TABLES IN silver;

In [ ]:
%sql
-- Describe the students table — verify the new phone_number column is present
DESCRIBE TABLE silver.students;

In [ ]:
# Show catalog details
display(spark.sql(f"DESCRIBE CATALOG EXTENDED {catalog}"))

In [ ]:
%sql 
-- Query tags via information schema (structured way)
SELECT *
FROM system.information_schema.catalog_tags
WHERE catalog_name = current_catalog();

---

## Next steps

You have completed all notebook exercises. Return to the **lab instructions** and continue with:

- **Exercise**: Configure an AI/BI Genie Space (optional)